![image info](https://raw.githubusercontent.com/albahnsen/MIAD_ML_and_NLP/main/images/banner_1.png)

# Reto opcional · Desplegar la Lambda con Docker

En clase vimos cómo disponibilizar un modelo de `scikit-learn` en AWS Lambda usando un **Layer**. Ese enfoque funciona, pero tiene un límite duro: el paquete de despliegue (código + layers) no puede superar **250 MB descomprimidos**. Para un modelo que, además de las librerías actuales, use `pandas`, o un modelo de NLP que use `transformers` y `torch`, ese límite se vuelve un problema real.

Existe un camino alternativo: **desplegar la Lambda como una imagen de Docker**. Con este enfoque el límite sube a **10 GB**, y la función puede incluir cualquier dependencia.

Este notebook es un **reto abierto**: no vas a encontrar aquí el paso a paso. La idea es que uses herramientas de AI (ChatGPT, Claude, Gemini) para construir tu propia solución. La habilidad que estamos practicando no es "desplegar una Lambda en Docker", sino **aprender a aprender** con AI.

## ¿Qué es Docker?

**Docker** es una herramienta que empaqueta una aplicación junto con todo lo que necesita para correr (sistema operativo base, librerías, dependencias, código y configuración) en una unidad portable llamada **imagen**. Cuando esa imagen se ejecuta, el resultado se llama **contenedor**.

La idea clave es: *"en mi máquina funciona"* deja de ser un problema. Si la imagen corre en tu laptop, corre idéntica en AWS, en un servidor de un compañero, o en cualquier otra parte. Todo el entorno está congelado dentro de la imagen.

Para este reto, vas a construir una imagen Docker que contenga Python 3.12, `scikit-learn`, `numpy`, tu modelo `.pkl` y tu `lambda_function.py`. AWS Lambda puede ejecutar esa imagen directamente como si fuera una función normal, sin necesidad de Layers.

## Objetivo

Toma el mismo modelo de phishing (`phishing.pkl`) y el mismo `lambda_function.py` que usamos con el Layer, y despliégalo como una **Lambda basada en imagen de Docker**, usando el siguiente flujo:

1. **GitHub**: el código fuente de tu Lambda vive en un repo. *Necesitarás: un repositorio con tu `lambda_function.py`, el `.pkl` del modelo, un `Dockerfile` y un `buildspec.yml`.*
2. **AWS CodeBuild**: toma el código del repo y construye la imagen de Docker. *Necesitarás: un proyecto de CodeBuild conectado a GitHub, un entorno de build con Docker habilitado, y un rol de IAM con los permisos correctos.*
3. **Amazon ECR**: almacena la imagen construida. *Necesitarás: un repositorio privado en ECR donde CodeBuild pueda hacer `push`.*
4. **AWS Lambda**: se crea a partir de la imagen en ECR. *Necesitarás: una función Lambda configurada como *container image* apuntando a tu URI en ECR.*

Al final debes tener una Lambda **funcionalmente idéntica** a la del ejercicio con Layer: la misma URL probada con un `{"url": "..."}` devuelve la misma probabilidad de phishing.

> 💡 **Sugerencia:** si no sabes qué archivos o recursos específicos requiere cada paso, **pregúntale al AI que te liste todo lo necesario antes de empezar**. Tener claro el inventario completo antes de escribir el primer comando te va a ahorrar horas de ida y vuelta.

## ⚠️ La parte más tediosa: permisos de IAM

Si algo va a fallar en este reto, lo más probable es que sea por **permisos de IAM mal configurados**. No porque sea difícil conceptualmente, sino porque son muchos servicios hablando entre sí, y cada uno necesita permisos explícitos para interactuar con los demás.

Estas son las relaciones que tienes que habilitar:

- **CodeBuild → ECR**: el rol de ejecución de CodeBuild debe poder autenticarse contra ECR y hacer `push` de imágenes.
- **CodeBuild → CloudWatch Logs**: para que puedas ver los logs del build cuando algo falle.
- **CodeBuild → GitHub**: vía OAuth o un *personal access token* para leer el repo.
- **Lambda → ECR**: el servicio Lambda debe poder leer la imagen del repositorio privado de ECR.
- **Tu usuario → todo lo anterior**: para crear y configurar cada uno de estos recursos.

Cuando algo falle, **lee el mensaje de error completo**, que casi siempre indica cuál permiso específico falta. Errores típicos: `AccessDeniedException`, `is not authorized to perform`, `User is not authorized`.

> 💡 Un buen primer prompt: *"Dame la lista completa de roles y políticas de IAM que necesito para desplegar una Lambda como imagen Docker usando CodeBuild y ECR. Explica qué hace cada uno."*

## Los 4 conceptos que debes investigar

No te voy a decir qué comandos correr. Te voy a decir qué conceptos tienes que entender. Investiga cada uno **antes** de pedirle código a un AI:

### 1. Dockerfile para Lambda
- AWS publica imágenes base oficiales para correr Lambdas. ¿Cuál es la imagen base correcta para Python 3.12?
- ¿Qué estructura de carpetas espera Lambda dentro del contenedor?
- ¿Cómo se especifica cuál es el handler (`lambda_function.lambda_handler`) desde el Dockerfile?

### 2. AWS CodeBuild
- ¿Qué es un `buildspec.yml` y qué fases tiene?
- ¿Cómo se conecta CodeBuild a un repo de GitHub?
- ¿Qué permisos de IAM necesita el proyecto de CodeBuild para escribir en ECR?

### 3. Amazon ECR
- ¿Cuál es la diferencia entre un *repository* y una *imagen* en ECR?
- ¿Cómo se autentica Docker contra ECR para poder hacer `push`?
- ¿Qué formato tiene la URI de una imagen en ECR?

### 4. Lambda desde imagen
- ¿Qué cambia al crear una Lambda desde imagen vs desde zip + layers?
- ¿Qué pasa cuando haces `push` de una nueva versión de la imagen? ¿La Lambda se actualiza sola?
- ¿Cómo se invoca y prueba igual que antes?

## Cómo conversar con un AI para este reto

La calidad de tu prompt determina la calidad del código que recibes. Unas reglas prácticas:

- **No pidas "hazme una Lambda en Docker".** Vas a recibir una respuesta genérica que no compila. Pide cosas específicas: *"escribe un Dockerfile para una Lambda Python 3.12 que tenga scikit-learn 1.7.2 instalado y cuyo handler sea lambda_function.lambda_handler"*.
- **Pregúntale al AI el *porqué*, no solo el *qué*.** Si te da un comando, pregúntale qué hace cada flag. Si no entiendes la respuesta, eres tú el que va a depurar cuando falle.
- **Cuando algo falla, pásale al AI el error exacto**, no lo resumas. Los mensajes de error de AWS son larguísimos por una razón.
- **Valida contra la documentación oficial.** Los AI a veces inventan flags o nombres de servicios. AWS tiene docs explícitas para Lambda-on-container: úsalas como verdad final.

### Preguntas guía para arrancar

Prueba empezar con alguna de estas y desde ahí construye la solución:

1. *"¿Por qué desplegar una Lambda como imagen de Docker evita el límite de 250 MB que tiene el despliegue con layers?"*
2. *"¿Cuál es la imagen base oficial de AWS para Lambda con Python 3.12 y dónde está documentada?"*
3. *"Dame un `buildspec.yml` mínimo que haga login a ECR, construya una imagen Docker y haga push. Explícame cada línea."*
4. *"¿Qué permisos IAM necesita el rol de CodeBuild para poder hacer push a ECR y qué política gestionada de AWS los agrupa?"*